# Reallocation and labour productivity (v1)

How job reallocation (LBD job flows) relates to labour productivity (ONS GVA per hour worked) across ITL2 regions. Split out of `analysis/reallocation/reallocation_v1.ipynb`, which has the reallocation definitions, trends and caveats.

- **Data:** `clean/itl2_2021/job_flows_itl2.csv` (cleaned by `build/job_flows.ipynb`) and `clean/itl2_2021/productivity_itl2.csv` (`build/productivity.ipynb`); ITL2 **2021** (41 regions).
- **Rates:** reallocation = (job creation + job destruction) / average employment, **pooled from counts** over each period.
- **Charts** saved to `analysis/productivity/charts/` (JSON + PNG). Shared chart setup: `analysis/chart_setup.py`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

BASE  = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
DIR21 = BASE / "clean" / "itl2_2021"
FIG   = BASE / "analysis" / "productivity" / "charts"
FIG.mkdir(parents=True, exist_ok=True)
sys.path.append(str(BASE / "analysis"))
from chart_setup import styles, alt, ECO, TXT, GREY, label_layers

reg = pd.read_csv(DIR21 / "job_flows_itl2.csv")            # all years, cleaned
jf = reg[reg.analysis_window].copy()                         # 2003-2024
print(f"{jf.itl2_code.nunique()} regions x {jf.year.nunique()} years ({jf.year.min()}-{jf.year.max()})")

41 regions x 22 years (2003-2024)


## 1. Reallocation and labour productivity (GVA per hour worked)

**Why care:** in the business dynamism literature, reallocation matters because it can move jobs from less to more productive firms (Foster, Haltiwanger & Krizan 2001; Decker et al. 2017, 2020). But a raw regional scatter of churn against productivity can mislead, so the comparison is built to avoid the usual traps:

| Pitfall | How it is handled here |
|---|---|
| Comparing productivity *levels* across regions with real (chained volume) indices (ONS advises against this) | **Levels:** current-price £ per hour **relative to the UK in the same year** (UK = 100), so inflation cancels. **Growth:** real (chained volume, 2022 £) GVA per hour, compared within each region over time. |
| Cyclical noise and the 2010/2023 spikes in the LBD | Rates **pooled over several years** from counts (Σ JC + Σ JD) / Σ average employment. Robustness checks drop 2020–21 and 2023 and use a pre-COVID window. |
| Levels ≠ growth: high-churn places may simply be rich places | Two separate questions: (a) do more productive regions churn more? (b) does initial reallocation **predict later** productivity growth? |
| Outliers (Inner London) and small regions | Pearson and Spearman correlations, with and without London, plus employment-weighted fits. |
| Peak-to-peak growth | Growth measured **2007 → 2019** (pre-GFC peak to pre-COVID), using reallocation in the preceding 2004–07 period. |
| Persistent regional differences (sector mix, city size) | Panel check with **region and year fixed effects**: does a region's reallocation in year *t* predict its own productivity growth over the following 1 and 3 years? Standard errors are clustered by region. |
| Sector composition (e.g. high-churn, low-productivity hospitality; London finance) | **Not addressed here.** It needs the LBD broken down by industry, inside the secure environment (see the caveats below). |

Northern Ireland is excluded wherever reallocation before 2011 is used (its early series is erratic; see `build/job_flows.ipynb`). Productivity comes from `clean/itl2_2021/productivity_itl2.csv` (hours from 2004, to 2023). Correlations are descriptive, not causal.

In [2]:
from scipy import stats

prod = pd.read_csv(DIR21 / "productivity_itl2.csv").query("hours_worked_weekly.notna()").copy()
prod["gva_cp"]   = prod.gva_per_hour_gbp * prod.hours_worked_weekly            # current-price GVA (per week; scale cancels)
prod["gva_real"] = prod.gva_per_hour_real_2022gbp * prod.hours_worked_weekly    # real GVA, 2022 £
uk_ph = prod.groupby("year").gva_cp.sum() / prod.groupby("year").hours_worked_weekly.sum()
prod["gva_at_uk_rate"] = prod.hours_worked_weekly * prod.year.map(uk_ph)        # what the region's hours would produce at the UK £/hour
LONDON = lambda d: d.itl2_code.str.startswith("TLI")

def ni_trim(d, ni_from):
    """Optionally drop Northern Ireland years before `ni_from` (erratic early job-flows series)."""
    return d if ni_from is None else d[~((d.itl2_code == "TLN0") & (d.year < ni_from))]

def pooled_realloc(years, drop=(), ni_from=None):
    """Pooled annual reallocation rate from counts: (sum JC + sum JD) / sum average employment."""
    d = ni_trim(jf[jf.year.between(*years) & ~jf.year.isin(drop)], ni_from)
    g = d.groupby(["itl2_code", "itl2_name"])[["total_jc", "total_jd", "emp_avg"]].sum()
    n_years = d.groupby(["itl2_code", "itl2_name"]).year.nunique()
    return pd.DataFrame({"realloc": 100 * (g.total_jc + g.total_jd) / g.emp_avg, "emp": g.emp_avg / n_years})

def prod_rel(years, drop=(), ni_from=None):
    """GVA per hour relative to the UK (UK = 100), pooled over years: sum of GVA / sum of (hours x UK £ per hour)."""
    d = ni_trim(prod[prod.year.between(*years) & ~prod.year.isin(drop)], ni_from)
    g = d.groupby("itl2_code")[["gva_cp", "gva_at_uk_rate"]].sum()
    return (100 * g.gva_cp / g.gva_at_uk_rate).rename("prod_rel")

def real_growth(y0, y1):
    """Annualised real growth in GVA per hour (% per year, log points)."""
    r = prod.pivot(index="itl2_code", columns="year", values="gva_per_hour_real_2022gbp")
    return (100 * np.log(r[y1] / r[y0]) / (y1 - y0)).rename("growth")

def ols(y, X, w=None, groups=None):
    """(Weighted) OLS with HC1 or cluster-robust standard errors. Returns coefficients and standard errors (constant first)."""
    X = np.column_stack([np.ones(len(y)), X]); w = np.ones(len(y)) if w is None else np.asarray(w, float)
    Xw, yw = X * np.sqrt(w)[:, None], y * np.sqrt(w)
    b = np.linalg.lstsq(Xw, yw, rcond=None)[0]; e = yw - Xw @ b
    A = np.linalg.pinv(Xw.T @ Xw); n, k = Xw.shape
    if groups is None:
        V = A @ ((Xw * e[:, None]).T @ (Xw * e[:, None])) @ A * n / (n - k)
    else:
        G = pd.factorize(groups)[0]; S = np.zeros((k, k))
        for g in np.unique(G):
            s = (Xw[G == g] * e[G == g][:, None]).sum(0); S += np.outer(s, s)
        ng = G.max() + 1; V = A @ S @ A * ng / (ng - 1) * (n - 1) / (n - k)
    return b, np.sqrt(np.diag(V))

# (a) levels: reallocation rate vs log GVA per hour relative to UK
lev_rows = []
def levels(label, years, drop=(), excl=None, weighted=False, ni_from=None):
    d = pooled_realloc(years, drop, ni_from).join(prod_rel(years, drop, ni_from), on="itl2_code").reset_index()
    d = d[~excl(d)] if excl is not None else d
    b, se = ols(d.realloc.values, np.log(d.prod_rel.values), d.emp if weighted else None)
    lev_rows.append({"specification": label, "n": len(d),
                     "pp per 10% higher productivity": b[1] * np.log(1.1), "se": se[1] * np.log(1.1),
                     "Pearson r": np.corrcoef(d.realloc, np.log(d.prod_rel))[0, 1], "Spearman rho": stats.spearmanr(d.realloc, d.prod_rel)[0]})
    return d

LEV = (2018, 2023)   # latest years with productivity data
lev = levels("2018–23, all regions", LEV)
levels("2018–23, employment-weighted", LEV, weighted=True)
levels("2018–23, excl. London (5 regions)", LEV, excl=LONDON)
levels("2018–23, excl. Inner London (2)", LEV, excl=lambda d: d.itl2_code.isin(["TLI3", "TLI4"]))
levels("2018–23, excl. 2020–21 and 2023", LEV, drop=(2020, 2021, 2023))
levels("2017–19 (pre-COVID)", (2017, 2019))
levels("2004–08, excl. NI", (2004, 2008), excl=lambda d: d.itl2_code == "TLN0")
levels("2004–08, excl. NI and London", (2004, 2008), excl=lambda d: (d.itl2_code == "TLN0") | LONDON(d))
FULL = (2004, 2023)  # whole overlap of job flows and productivity; NI from 2011
full = levels("2004–23 whole period (NI 2011–23)", FULL, ni_from=2011)
levels("2004–23, excl. London", FULL, ni_from=2011, excl=LONDON)
levels("2004–23, excl. spike years 2010, 2023 and COVID 2020–21", FULL, drop=(2010, 2020, 2021, 2023), ni_from=2011)
levels("2004–23, employment-weighted", FULL, ni_from=2011, weighted=True)
print("(a) Levels: reallocation rate on log GVA per hour (UK = 100); HC1 standard errors")
print(pd.DataFrame(lev_rows).round(2).to_string(index=False))

# (b) growth: initial reallocation vs subsequent real productivity growth (NI excluded throughout)
gr_rows = []
def growth_spec(label, ry, gy, excl=None, control=False):
    d = pooled_realloc(ry).join(real_growth(*gy), on="itl2_code").join(prod_rel(ry), on="itl2_code").reset_index()
    d = d[d.itl2_code != "TLN0"]; d = d[~excl(d)] if excl is not None else d
    X = np.column_stack([d.realloc, np.log(d.prod_rel)]) if control else d.realloc.values
    b, se = ols(d.growth.values, X)
    gr_rows.append({"specification": label, "n": len(d), "growth pp/yr per 10pp reallocation": 10 * b[1], "se": 10 * se[1],
                    "Pearson r": np.corrcoef(d.realloc, d.growth)[0, 1], "Spearman rho": stats.spearmanr(d.realloc, d.growth)[0]})
    return d

gro = growth_spec("Reallocation 2004–07 → growth 2007–19", (2004, 2007), (2007, 2019))
growth_spec("  + control for initial productivity (UK = 100)", (2004, 2007), (2007, 2019), control=True)
growth_spec("  excl. London", (2004, 2007), (2007, 2019), excl=LONDON)
growth_spec("Reallocation 2004–07 → growth 2007–23", (2004, 2007), (2007, 2023))
growth_spec("Reallocation 2004–08 → growth 2008–19", (2004, 2008), (2008, 2019))
growth_spec("Reallocation 2011–15 → growth 2015–23", (2011, 2015), (2015, 2023))
print("\n(b) Growth: annualised real GVA per hour growth on initial reallocation; NI excluded; HC1 standard errors")
print(pd.DataFrame(gr_rows).round(2).to_string(index=False))

# (c) panel: region's own reallocation in year t vs its real productivity growth t -> t+h, region + year fixed effects
pan = (jf[["itl2_code", "year", "reallocation_rate"]]
         .merge(prod[["itl2_code", "year", "gva_per_hour_real_2022gbp"]], on=["itl2_code", "year"])
         .query("not (itl2_code == 'TLN0' and year < 2011)").sort_values(["itl2_code", "year"]))
print("\n(c) Panel with region and year fixed effects; standard errors clustered by region")
for h in (1, 3):
    pan[f"g{h}"] = 100 * (np.log(pan.groupby("itl2_code").gva_per_hour_real_2022gbp.shift(-h)) - np.log(pan.gva_per_hour_real_2022gbp)) / h
    d = pan.dropna(subset=[f"g{h}"])
    X = np.column_stack([d.reallocation_rate, pd.get_dummies(d.itl2_code, drop_first=True, dtype=float),
                         pd.get_dummies(d.year, drop_first=True, dtype=float)])
    b, se = ols(d[f"g{h}"].values, X, groups=d.itl2_code.values)
    print(f"  growth over next {h} year(s): {10 * b[1]:+.2f} pp/yr per 10pp reallocation (se {10 * se[1]:.2f}); N = {len(d)}, {d.itl2_code.nunique()} regions")

(a) Levels: reallocation rate on log GVA per hour (UK = 100); HC1 standard errors
                                          specification  n  pp per 10% higher productivity   se  Pearson r  Spearman rho
                                   2018–23, all regions 41                            1.42 0.29       0.66          0.48
                           2018–23, employment-weighted 41                            1.58 0.28       0.66          0.48
                      2018–23, excl. London (5 regions) 36                            0.54 0.27       0.25          0.28
                        2018–23, excl. Inner London (2) 39                            0.88 0.30       0.40          0.40
                        2018–23, excl. 2020–21 and 2023 41                            1.51 0.32       0.63          0.45
                                    2017–19 (pre-COVID) 41                            1.65 0.31       0.66          0.54
                                      2004–08, excl. NI 40             

In [3]:
# chart (a): reallocation vs GVA per hour (levels), as a function so the same design serves several periods
SRC2 = "Sources: ONS Longitudinal Business Database (Secure Access, SN 9501); ONS Subregional productivity (June 2025); authors' calculations."

def fit_line(dd):
    b, _ = ols(dd.realloc.values, dd.prod_rel.values)
    x = np.linspace(dd.prod_rel.min(), dd.prod_rel.max(), 50)
    return pd.DataFrame({"x": x, "y": b[0] + b[1] * x})

def levels_chart(lv, years, period_label, title, note="", ni_from=None):
    d = lv.assign(group=np.where(LONDON(lv), "London", "Rest of UK"))
    j = ni_trim(jf[jf.year.between(*years)], ni_from)
    uk_realloc = 100 * j[["total_jc", "total_jd"]].sum().sum() / j.emp_avg.sum()
    r_all = np.corrcoef(d.realloc, d.prod_rel)[0, 1]
    r_exl = np.corrcoef(d[d.group == "Rest of UK"].realloc, d[d.group == "Rest of UK"].prod_rel)[0, 1]
    xmax = float(np.ceil((d.prod_rel.max() + 3) / 10) * 10)
    XS2 = alt.Scale(domain=[70, xmax], zero=False, nice=False); YS2 = alt.Scale(domain=[20, 40], zero=False, nice=False)
    fits = pd.concat([fit_line(d).assign(fit="Fit: all regions"), fit_line(d[d.group == "Rest of UK"]).assign(fit="Fit: excl. London")])
    lab = pd.concat([d.nlargest(2, "prod_rel"), d[d.group == "Rest of UK"].nlargest(1, "prod_rel")]).drop_duplicates("itl2_code")
    low = d.nsmallest(1, "realloc")   # labelled to the left: neighbouring points tend to sit to its right
    ref = lambda **kw: alt.Chart(pd.DataFrame({"v": [list(kw.values())[0]]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5)
    chart = alt.layer(
        ref(x=100.0).encode(x=alt.X("v:Q", scale=XS2)),
        ref(y=uk_realloc).encode(y=alt.Y("v:Q", scale=YS2)),
        alt.Chart(pd.DataFrame({"x": [100.0], "y": [YS2.domain[1]]})).mark_text(align="left", baseline="top", dx=4, dy=4, fontSize=10, color=TXT)
           .encode(x=alt.X("x:Q", scale=XS2), y=alt.Y("y:Q", scale=YS2), text=alt.value("UK = 100")),
        alt.Chart(pd.DataFrame({"x": [XS2.domain[0]], "y": [uk_realloc]})).mark_text(align="left", baseline="bottom", dx=4, dy=-3, fontSize=10, color=TXT)
           .encode(x=alt.X("x:Q", scale=XS2), y=alt.Y("y:Q", scale=YS2), text=alt.value(f"UK {uk_realloc:.1f}%")),
        alt.Chart(fits).mark_line(color=ECO["blue-dark"], strokeWidth=1.5).encode(
            x=alt.X("x:Q", scale=XS2), y=alt.Y("y:Q", scale=YS2),
            strokeDash=alt.StrokeDash("fit:N", scale=alt.Scale(domain=["Fit: all regions", "Fit: excl. London"], range=[[1, 0], [6, 4]]),
                                      legend=alt.Legend(title=None, orient="bottom"))),
        alt.Chart(d).mark_circle(size=70, opacity=0.9, stroke="white", strokeWidth=1).encode(
            x=alt.X("prod_rel:Q", scale=XS2, title="GVA per hour worked (UK = 100)", axis=alt.Axis(values=list(range(70, int(xmax) + 1, 10)))),
            y=alt.Y("realloc:Q", scale=YS2, title="Reallocation rate (% of employment)"),
            color=alt.Color("group:N", scale=alt.Scale(domain=["London", "Rest of UK"], range=[ECO["red"], ECO["blue-light"]]),
                            legend=alt.Legend(title=None, orient="bottom")),
            tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("prod_rel:Q", format=".0f", title="GVA/hour (UK=100)"),
                     alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
        *label_layers(lab, "prod_rel", "realloc", XS2, YS2, 540, 380),
        alt.Chart(low).mark_text(align="right", dx=-8, fontSize=10, color=TXT).encode(
            x=alt.X("prod_rel:Q", scale=XS2), y=alt.Y("realloc:Q", scale=YS2), text="itl2_name:N"),
    ).resolve_scale(color="independent", strokeDash="independent").properties(title=alt.Title(title,
        subtitle=[f"ITL2 regions, {period_label} pooled. Correlation r = {r_all:.2f} across all regions; {r_exl:.2f} excluding London.",
                  f"Lines are linear fits. Productivity is current-price GVA per hour relative to the UK in the same year.{note}", SRC2]))
    return chart

lp = levels_chart(lev, LEV, "2018–23", "More productive regions churn more, but London drives most of the link")
styles.save(lp, str(FIG), "reallocation_productivity_levels", width=540, height=380)
lp.properties(width=540, height=380)

alt.LayerChart(...)

In [4]:
# chart (b): initial reallocation (2004-07) vs real productivity growth (2007-19)
g = gro.assign(group=np.where(LONDON(gro), "London", "Rest of UK"))
ukr = prod.groupby("year")[["gva_real", "hours_worked_weekly"]].sum()
uk_g = 100 * np.log((ukr.gva_real / ukr.hours_worked_weekly)[2019] / (ukr.gva_real / ukr.hours_worked_weekly)[2007]) / 12
uk_r0 = 100 * jf[jf.year.between(2004, 2007)][["total_jc", "total_jd"]].sum().sum() / jf[jf.year.between(2004, 2007)].emp_avg.sum()
r_g, rho_g = gr_rows[0]["Pearson r"], gr_rows[0]["Spearman rho"]
XS3 = alt.Scale(domain=[float(np.floor(g.realloc.min() - 1)), float(np.ceil(g.realloc.max() + 1))], zero=False, nice=False)
YS3 = alt.Scale(domain=[float(np.floor(g.growth.min() * 4 - 1) / 4), float(np.ceil(g.growth.max() * 4 + 1) / 4)], zero=False, nice=False)
b, _ = ols(g.growth.values, g.realloc.values)
fit = pd.DataFrame({"x": XS3.domain, "y": [b[0] + b[1] * x for x in XS3.domain]})
labg = pd.concat([g.nlargest(2, "realloc"), g.nlargest(1, "growth"), g.nsmallest(1, "growth")]).drop_duplicates("itl2_code")

gp = alt.layer(
    alt.Chart(pd.DataFrame({"v": [uk_r0]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(x=alt.X("v:Q", scale=XS3)),
    alt.Chart(pd.DataFrame({"v": [uk_g]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YS3)),
    alt.Chart(pd.DataFrame({"x": [uk_r0], "y": [YS3.domain[1]]})).mark_text(align="left", baseline="top", dx=4, dy=4, fontSize=10, color=TXT)
       .encode(x=alt.X("x:Q", scale=XS3), y=alt.Y("y:Q", scale=YS3), text=alt.value(f"UK {uk_r0:.1f}%")),
    alt.Chart(pd.DataFrame({"x": [XS3.domain[1]], "y": [uk_g]})).mark_text(align="right", baseline="bottom", dx=-4, dy=-3, fontSize=10, color=TXT)
       .encode(x=alt.X("x:Q", scale=XS3), y=alt.Y("y:Q", scale=YS3), text=alt.value(f"UK {uk_g:.2f}% a year")),
    alt.Chart(fit).mark_line(color=ECO["blue-dark"], strokeWidth=1.5).encode(x=alt.X("x:Q", scale=XS3), y=alt.Y("y:Q", scale=YS3)),
    alt.Chart(g).mark_circle(size=70, opacity=0.9, stroke="white", strokeWidth=1).encode(
        x=alt.X("realloc:Q", scale=XS3, title="Reallocation rate, 2004–07 (% of employment)"),
        y=alt.Y("growth:Q", scale=YS3, title="Real GVA per hour growth, 2007–19 (% per year)"),
        color=alt.Color("group:N", scale=alt.Scale(domain=["London", "Rest of UK"], range=[ECO["red"], ECO["blue-light"]]),
                        legend=alt.Legend(title=None, orient="bottom")),
        tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("realloc:Q", format=".1f", title="Reallocation 2004–07 (%)"),
                 alt.Tooltip("growth:Q", format=".2f", title="Growth 2007–19 (%/yr)")]),
    *label_layers(labg, "realloc", "growth", XS3, YS3, 540, 380),
).properties(title=alt.Title("Regions that churned more before the financial crisis did not see faster productivity growth",
    subtitle=[f"ITL2 regions; reallocation pooled over 2004–07, growth measured peak to peak (2007–19). r = {r_g:.2f}.",
              "Real growth from ONS chained-volume GVA per hour. Northern Ireland excluded (erratic early series).", SRC2]))
styles.save(gp, str(FIG), "reallocation_productivity_growth", width=540, height=380)
gp.properties(width=540, height=380)

alt.LayerChart(...)

### 1c. Whole period, 2004–2023

The same levels comparison, pooled over the whole span covered by both datasets (productivity hours start in 2004 and end in 2023). Northern Ireland enters from 2011 only, for both measures. This is the most stable cross-regional picture but mixes the pre-crisis boom, the 2010s and COVID. The robustness rows `2004–23 …` in table (a) drop the spike/COVID years and London. Because UK-wide churn was higher in 2004–10 than afterwards, NI's 2011–23 rate is not strictly comparable with the other regions' 2004–23 rates (it will look somewhat lower). Even so, NI is below the UK in 2011–23 alone.

In [5]:
fp = levels_chart(full, FULL, "2004–23", "Over two decades, more productive regions have churned more, led by London",
                  note=" Northern Ireland 2011–23.", ni_from=2011)
styles.save(fp, str(FIG), "reallocation_productivity_levels_2004_2023", width=540, height=380)
fp.properties(width=540, height=380)

alt.LayerChart(...)

### 1d. Three periods side by side (after the ONS chart)

The same levels comparison for three five-year periods, **2004–08**, **2010–14** and **2019–23**, drawn as three panels with **shared axes** so shifts between periods are visible. It follows an ONS chart, which noted that reallocation fell everywhere except London and that London's outlier position creates much of the productivity–reallocation correlation.

- x, two versions: **real GVA per hour in 2022 £** (`reallocation_productivity_periods_real`; pooled Σ real GVA ÷ Σ hours), which shows *movement over time* (the post-2008 productivity stall: UK ≈ £37 → £38 → £40), and **GVA per hour relative to the UK** (UK = 100, `reallocation_productivity_periods`), which isolates each region's *relative position*. Within-period correlations are essentially the same in both. Real regional levels rely on ONS chained-volume indices anchored in 2022, so cross-region gaps in early years are approximate. A third version, `reallocation_productivity_periods_gdpdefl`, deflates current-price GVA per hour by the **UK GDP deflator** (ONS YBGB, 2022 = 100): regional gaps are exact in every period (as in UK = 100), and movement over time is real but uses one UK-wide price index rather than industry-specific ones. Both £ versions share the same x-axis.
- y: pooled reallocation rate; dot size = average employment; London in red; dashed line = linear fit across all regions.
- Labels are placed greedily (London and outlying regions first), allowing for dot size, and skipped where they would collide with another label or dot.
- **Caveats:** 2010–14 contains the 2010 LBD spike and 2019–23 the 2023 spike and COVID. The printed check below repeats the correlations without those years. Northern Ireland is excluded from 2004–08 and enters 2010–14 from 2011.

**Regular windows (no gaps).** The three windows above skip 2009 and 2015–18, so their spacing is uneven. Two gap-free versions in real 2022 £: **A** = four 5-year blocks (2004–08, 2009–13, 2014–18, 2019–23; `reallocation_productivity_windows_A`) and **C** = three contiguous blocks (2004–10, 2011–17, 2018–23; `..._C`). Both show steadily falling UK reallocation (A: 32.5 → 30.1 → 28.4 → 27.5%) alongside slow productivity growth (£37 → £40). The correlation *excluding London* does **not** fall steadily: A gives 0.41, 0.23, 0.36, 0.23 and C gives 0.37, 0.37, 0.25. So "London increasingly drives the link" depends on which windows are chosen. The robust statement is that outside London the link is weak in every period.

In [6]:
PANELS = {"2004–08": (2004, 2008), "2010–14": (2010, 2014), "2019–23": (2019, 2023)}
NI_FROM = {"2004–08": 9999, "2010–14": 2011, "2019–23": None}          # 9999 = NI excluded
SPIKES = (2010, 2020, 2021, 2023)
def prod_gbp(years, ni_from=None):
    """Real GVA per hour (2022 £), pooled: sum of real GVA / sum of hours."""
    d = ni_trim(prod[prod.year.between(*years)], ni_from)
    g = d.groupby("itl2_code")[["gva_real", "hours_worked_weekly"]].sum()
    return (g.gva_real / g.hours_worked_weekly).rename("prod_gbp")

# GDP deflator (ONS YBGB, implied deflator at market prices), rebased to 2022 = 100; downloaded once to raw/prices/
import urllib.request
DEFL = BASE / "raw" / "prices" / "gdp_deflator_ybgb.csv"
if not DEFL.exists():
    DEFL.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request("https://www.ons.gov.uk/generator?format=csv&uri=/economy/grossdomesticproductgdp/timeseries/ybgb/qna",
                                 headers={"User-Agent": "Mozilla/5.0"})
    DEFL.write_bytes(urllib.request.urlopen(req, timeout=60).read())
gdpd = pd.read_csv(DEFL, header=None, names=["period", "value"], dtype=str)
gdpd = gdpd[gdpd.period.str.fullmatch(r"\d{4}")].astype({"period": int, "value": float}).set_index("period").value
gdpd = 100 * gdpd / gdpd[2022]
prod["gva_defl"] = prod.gva_cp * 100 / prod.year.map(gdpd)        # current-price GVA in 2022 prices, one UK-wide deflator

def prod_defl(years, ni_from=None):
    """GVA per hour, current prices deflated by the GDP deflator (2022 £), pooled: sum deflated GVA / sum hours."""
    d = ni_trim(prod[prod.year.between(*years)], ni_from)
    g = d.groupby("itl2_code")[["gva_defl", "hours_worked_weekly"]].sum()
    return (g.gva_defl / g.hours_worked_weekly).rename("prod_defl")

def build_panels(PANELS, NI_FROM):
    pan_rows, panels = [], {}
    for lab, yrs in PANELS.items():
        d = pooled_realloc(yrs, ni_from=NI_FROM[lab]).join(prod_rel(yrs, ni_from=NI_FROM[lab]), on="itl2_code").reset_index().dropna()
        d = d.join(prod_gbp(yrs, NI_FROM[lab]), on="itl2_code").join(prod_defl(yrs, NI_FROM[lab]), on="itl2_code").assign(period=lab, group=np.where(LONDON(d), "London", "Rest of UK"))
        pj = ni_trim(prod[prod.year.between(*yrs)], NI_FROM[lab])
        uk_gbp = pj.gva_real.sum() / pj.hours_worked_weekly.sum()
        uk_defl = pj.gva_defl.sum() / pj.hours_worked_weekly.sum()
        j = ni_trim(jf[jf.year.between(*yrs)], NI_FROM[lab])
        uk = 100 * j[["total_jc", "total_jd"]].sum().sum() / j.emp_avg.sum()
        ds = pooled_realloc(yrs, SPIKES, NI_FROM[lab]).join(prod_rel(yrs, SPIKES, NI_FROM[lab]), on="itl2_code").dropna()
        r = lambda x: np.corrcoef(x.realloc, x.prod_rel)[0, 1]          # linear, matching the chart lines
        rest = d[d.group == "Rest of UK"]
        pan_rows.append({"period": lab, "n": len(d), "UK reallocation": uk, "UK GVA/hour (2022 £, chained volume)": uk_gbp, "UK GVA/hour (2022 £, GDP deflator)": uk_defl,
                         "London (pooled)": 100 * j[LONDON(j)][["total_jc", "total_jd"]].sum().sum() / j[LONDON(j)].emp_avg.sum(),
                         "UK excl. London": 100 * j[~LONDON(j)][["total_jc", "total_jd"]].sum().sum() / j[~LONDON(j)].emp_avg.sum(),
                         "r all": r(d), "r excl. London": r(rest), "r all, excl. 2010/20/21/23": r(ds)})
        panels[lab] = (d, uk, r(d), r(rest), uk_gbp, uk_defl)
    return panels, pd.DataFrame(pan_rows)

panels, pan_tab = build_panels(PANELS, NI_FROM)
print(pan_tab.round(2).to_string(index=False))

ALWAYS_LABEL = ["Greater Manchester"]     # labelled in every panel even if it has to overlap a dot

def place_labels(d, xs, ys, width, height, fs=9, size=None, xcol="prod_rel", max_labels=None, max_len=None, force=True, pad=1):
    """Greedy labels in priority order (ALWAYS_LABEL, then London, then outlying regions). Tries right, left, above and
    below the dot; skips the label if every position hits a placed label, another dot or the frame, except ALWAYS_LABEL
    regions, which take the first position that avoids other labels."""
    (x0, x1), (y0, y1) = xs.domain, ys.domain
    d = d.copy()
    d["px"] = (d[xcol] - x0) / (x1 - x0) * width
    d["py"] = (y1 - d.realloc) / (y1 - y0) * height
    d["short"] = d.itl2_name.str.replace(" - ", " ", regex=False).str.replace(" and ", " & ", regex=False)
    z = lambda s: (s - s.mean()) / s.std()
    d["prio"] = (np.where(d.itl2_name.isin(ALWAYS_LABEL), 100, 0) + np.where(d.group == "London", 10, 0)
                 + np.hypot(z(np.log(d[xcol])), z(d.realloc)))
    (s0, s1), (a0, a1) = size.domain, size.range                      # dot radius from the (linear, area) size scale
    d["rad"] = np.sqrt((a0 + (d.emp - s0) / (s1 - s0) * (a1 - a0)) / np.pi)
    pts, boxes, out = d[["px", "py", "rad"]].values, [], []
    if max_len:                                                          # long names are left unlabelled (tooltips still show them)
        d = d[(d.short.str.len() <= max_len) | (d.group == "London") & (d.short.str.len() <= max_len + 4)]
    for _, r in d.sort_values("prio", ascending=False).iterrows():
        if max_labels and len(out) >= max_labels:
            break
        w, h, gap = 0.56 * fs * len(r.short), fs + 2, r.rad + 3
        cands = [("left", gap, 0.0), ("right", -gap, 0.0), ("center", 0.0, -(gap + h / 2)), ("center", 0.0, gap + h / 2)]
        if not force:                                                   # sparse mode: beside the dot only (above/below reads ambiguously)
            cands = cands[:2]
        fallback = None
        for align, dx, dy in cands:
            bx0 = r.px + dx if align == "left" else (r.px + dx - w if align == "right" else r.px - w / 2)
            b = (bx0, r.py + dy - h / 2, bx0 + w, r.py + dy + h / 2)
            inside = b[0] >= 0 and b[2] <= width and b[1] >= 0 and b[3] <= height
            hits_lab = any(not (b[2] < o[0] or b[0] > o[2] or b[3] < o[1] or b[1] > o[3]) for o in boxes)
            hits_pt = any(b[0] - rr - pad <= px <= b[2] + rr + pad and b[1] - rr - pad <= py <= b[3] + rr + pad
                          for px, py, rr in pts if (px, py) != (r.px, r.py))
            pick = {**r.to_dict(), "align": align, "dx": float(dx), "dy": float(dy)}
            if inside and not hits_lab and not hits_pt:
                boxes.append(b); out.append(pick); fallback = None; break
            if inside and not hits_lab and fallback is None:
                fallback = (b, pick)
        else:
            if force and r.itl2_name in ALWAYS_LABEL and fallback:
                boxes.append(fallback[0]); out.append(fallback[1])
    return pd.DataFrame(out)

allp = pd.concat([v[0] for v in panels.values()])
YS4 = alt.Scale(domain=[float(np.floor(allp.realloc.min() - 1)), float(np.ceil(allp.realloc.max() + 1))], zero=False, nice=False)
SIZE = alt.Scale(domain=[0, float(allp.emp.max())], range=[8, 200])
W4, H4 = 330, 340
XMODES = {   # x column, axis title, scale, tick step, UK reference, tooltip format
    "rel":  dict(col="prod_rel", title="GVA per hour worked (UK = 100)", step=20, fmt=".0f",
                 scale=alt.Scale(domain=[70, float(np.ceil((allp.prod_rel.max() + 3) / 10) * 10)], zero=False, nice=False),
                 uk=lambda lab: 100.0, uk_lab=lambda lab: "UK = 100"),
    "real": dict(col="prod_gbp", title="GVA per hour worked (£, 2022 prices)", step=10, fmt=".1f",
                 scale=alt.Scale(domain=[float(np.floor((allp.prod_gbp.min() - 2) / 5) * 5), float(np.ceil((allp.prod_gbp.max() + 3) / 5) * 5)], zero=False, nice=False),
                 uk=lambda lab: panels[lab][4], uk_lab=lambda lab: f"UK £{panels[lab][4]:.0f}"),
    "defl": dict(col="prod_defl", title="GVA per hour worked (£, 2022 prices, GDP deflator)", step=10, fmt=".1f",
                 scale=alt.Scale(domain=[float(np.floor((min(allp.prod_gbp.min(), allp.prod_defl.min()) - 2) / 5) * 5),
                                         float(np.ceil((max(allp.prod_gbp.max(), allp.prod_defl.max()) + 3) / 5) * 5)], zero=False, nice=False),
                 uk=lambda lab: panels[lab][5], uk_lab=lambda lab: f"UK £{panels[lab][5]:.0f}"),
}
XMODES["real"]["scale"] = XMODES["defl"]["scale"]     # same x-axis for both £ versions, so they can be compared directly

def panel(lab, first, mode, panels=panels, W=W4, lab_opts=None, two_fits=False):
    m = XMODES[mode]; XS = m.get("xs") or m["scale"]; xc = m["col"]
    d, uk, r_all, r_ex, *_ = panels[lab]
    fx = np.linspace(d[xc].min(), d[xc].max(), 50); b, _ = ols(d.realloc.values, d[xc].values)
    labs = place_labels(d, XS, YS4, W, H4, size=SIZE, xcol=xc, **(lab_opts or {}))
    ax_y = alt.Axis(title="Reallocation rate (% of employment)" if first else None, labels=first, ticks=first, labelExpr="datum.label + '%'")
    ukx = {"rel": 100.0, "real": panels[lab][4], "defl": panels[lab][5]}[mode]
    uk_lab = "UK = 100" if mode == "rel" else f"UK £{ukx:.0f}"
    layers = [
        alt.Chart(pd.DataFrame({"v": [ukx]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(x=alt.X("v:Q", scale=XS)),
        alt.Chart(pd.DataFrame({"v": [uk]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YS4, axis=ax_y)),
        alt.Chart(pd.DataFrame({"x": [XS.domain[1]], "y": [uk]})).mark_text(align="right", baseline="bottom", dx=-3, dy=-3, fontSize=9, color=TXT)
           .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS4), text=alt.value(f"UK {uk:.1f}%")),
        alt.Chart(pd.DataFrame({"x": [ukx], "y": [YS4.domain[1]]})).mark_text(align="left", baseline="top", dx=3, dy=3, fontSize=9, color=TXT)
           .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS4), text=alt.value(uk_lab)),
        alt.Chart(pd.DataFrame({"x": fx, "y": b[0] + b[1] * fx})).mark_line(**(dict(color=ECO["red"], strokeWidth=1.6, opacity=0.65) if two_fits else dict(color=ECO["turquoise"], strokeWidth=2, strokeDash=[6, 4])))
           .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS4)),
        *([fit_layer(d[d.group == "Rest of UK"], xc, XS, ECO["blue-dark"])] if two_fits else []),
        alt.Chart(d).mark_circle(opacity=0.85, stroke="white", strokeWidth=0.6).encode(
            x=alt.X(f"{xc}:Q", scale=XS, title=m["title"], axis=alt.Axis(values=list(range(int(XS.domain[0]), int(XS.domain[1]) + 1, m["step"])),
                                                                    labelExpr="datum.label" if mode == "rel" else "'£' + datum.label")),
            y=alt.Y("realloc:Q", scale=YS4, axis=ax_y),
            size=alt.Size("emp:Q", scale=SIZE, legend=None),
            color=alt.Color("group:N", scale=alt.Scale(domain=["London", "Rest of UK"], range=[ECO["red"], ECO["blue-light"]]),
                            legend=alt.Legend(title=None, orient="bottom") if first else None),
            order=alt.Order("emp:Q", sort="descending"),
            tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip(f"{xc}:Q", format=m["fmt"], title="GVA per hour (£ or UK=100)"),
                     alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)"), alt.Tooltip("emp:Q", format=",.0f", title="Employment")]),
        *[alt.Chart(labs.iloc[[i]]).mark_text(align=r["align"], baseline="middle", dx=r["dx"], dy=r["dy"], fontSize=9, color=TXT)
            .encode(x=alt.X(f"{xc}:Q", scale=XS), y=alt.Y("realloc:Q", scale=YS4), text="short:N") for i, r in labs.iterrows()],
    ]
    return alt.layer(*layers).properties(width=W, height=H4,
        title=alt.Title(f"{lab}", subtitle=f"r = {r_all:.2f} (excl. London {r_ex:.2f})", anchor="start", fontSize=13, subtitleFontSize=11))

def fit_layer(dd, xc, XS, colour):
    """Thin, semi-transparent straight-line fit across the x-range of dd."""
    bb, _ = ols(dd.realloc.values, dd[xc].values); xx = np.linspace(dd[xc].min(), dd[xc].max(), 50)
    return (alt.Chart(pd.DataFrame({"x": xx, "y": bb[0] + bb[1] * xx})).mark_line(color=colour, strokeWidth=1.6, opacity=0.65)
              .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS4)))

def periods_chart(mode, title, note, panels=panels, W=W4, ni_note="Northern Ireland excluded in 2004–08 and from 2011 in 2010–14.", lab_opts=None, two_fits=False):
    fits = ("red line = linear fit across all regions, navy line = fit excluding London." if two_fits else "dashed line = linear fit.")
    return alt.hconcat(*[panel(lab, i == 0, mode, panels, W, lab_opts, two_fits) for i, lab in enumerate(panels)], spacing=12).properties(title=alt.Title(title,
        subtitle=[f"Reallocation rate against GVA per hour by ITL2 region, pooled over each period. Dot size = employment; {fits}",
                  f"{note} {ni_note}", SRC2])
    ).resolve_scale(color="shared", size="shared")

pc_rel = periods_chart("rel", "Reallocation has fallen faster outside London, so London increasingly drives the link with productivity",
                       "Productivity relative to the UK in the same years.")
styles.save(pc_rel, str(FIG), "reallocation_productivity_periods", width=W4, height=H4)
pc_real = periods_chart("real", "Churn fell across the UK while productivity barely moved after the financial crisis",
                        "Real GVA per hour in 2022 prices (ONS chained-volume measures). Correlations (r) are within each period.")
styles.save(pc_real, str(FIG), "reallocation_productivity_periods_real", width=W4, height=H4)
pc_defl = periods_chart("defl", "Churn fell across the UK while productivity barely moved after the financial crisis",
                        "Current-price GVA per hour deflated by the UK GDP deflator (2022 prices): regional gaps exact in each period.")
styles.save(pc_defl, str(FIG), "reallocation_productivity_periods_gdpdefl", width=W4, height=H4)

# Regular windows (no gaps): A = four contiguous 5-year blocks; C = three contiguous blocks (7, 7, 6 years). Real 2022 £ (chained volume).
WINDOW_SETS = {
    "A": ({"2004–08": (2004, 2008), "2009–13": (2009, 2013), "2014–18": (2014, 2018), "2019–23": (2019, 2023)},
          {"2004–08": 9999, "2009–13": 2011, "2014–18": None, "2019–23": None}, 255,
          "Northern Ireland excluded in 2004–08 and from 2011 in 2009–13.",
          dict(max_labels=5, max_len=20, force=False, pad=4), False),        # narrow panels: fewer, shorter, well-clear labels
    "C": ({"2004–10": (2004, 2010), "2011–17": (2011, 2017), "2018–23": (2018, 2023)},
          {"2004–10": 9999, "2011–17": None, "2018–23": None}, W4,
          "Northern Ireland excluded in 2004–10.", dict(max_labels=5, max_len=20, force=False, pad=4), True),
}
win_charts = {}
for key, (wins, ni, W, ni_note, lab_opts, two_fits) in WINDOW_SETS.items():
    pw, tab = build_panels(wins, ni)
    print(f"\nWindow set {key}:"); print(tab.round(2).to_string(index=False))
    win_charts[key] = periods_chart("real", "Churn fell across the UK while productivity barely moved after the financial crisis",
                                    "Real GVA per hour in 2022 prices (ONS chained-volume measures). Correlations (r) are within each period.",
                                    panels=pw, W=W, ni_note=ni_note, lab_opts=lab_opts, two_fits=two_fits)
    styles.save(win_charts[key], str(FIG), f"reallocation_productivity_windows_{key}", width=W, height=H4)
win_charts["A"]

 period  n  UK reallocation  UK GVA/hour (2022 £, chained volume)  UK GVA/hour (2022 £, GDP deflator)  London (pooled)  UK excl. London  r all  r excl. London  r all, excl. 2010/20/21/23
2004–08 40            32.50                                 36.98                               38.36            36.45            31.77   0.73            0.41                        0.73
2010–14 41            29.80                                 38.15                               38.64            35.25            28.75   0.72            0.28                        0.74
2019–23 41            27.53                                 40.48                               41.03            33.30            26.29   0.67            0.23                        0.62



Window set A:
 period  n  UK reallocation  UK GVA/hour (2022 £, chained volume)  UK GVA/hour (2022 £, GDP deflator)  London (pooled)  UK excl. London  r all  r excl. London  r all, excl. 2010/20/21/23
2004–08 40            32.50                                 36.98                               38.36            36.45            31.77   0.73            0.41                        0.73
2009–13 41            30.11                                 37.88                               38.62            35.00            29.18   0.66            0.23                        0.67
2014–18 41            28.37                                 38.97                               39.56            34.88            27.03   0.74            0.36                        0.74
2019–23 41            27.53                                 40.48                               41.03            33.30            26.29   0.67            0.23                        0.62



Window set C:
 period  n  UK reallocation  UK GVA/hour (2022 £, chained volume)  UK GVA/hour (2022 £, GDP deflator)  London (pooled)  UK excl. London  r all  r excl. London  r all, excl. 2010/20/21/23
2004–10 40            32.12                                 37.14                               38.51            36.02            31.40   0.72            0.37                        0.74
2011–17 41            28.62                                 38.52                               39.00            34.59            27.43   0.77            0.37                        0.77
2018–23 41            27.85                                 40.32                               40.89            33.90            26.56   0.68            0.25                        0.65


alt.HConcatChart(...)

In [7]:
# Window set C without London: the 36 other regions, tighter axes and more labels
wins_c, ni_c = WINDOW_SETS["C"][0], WINDOW_SETS["C"][1]
pw_c, _ = build_panels(wins_c, ni_c)
nl_panels = {lab: v[0][v[0].group == "Rest of UK"].copy() for lab, v in pw_c.items()}
allnl = pd.concat(nl_panels.values())
XSN = alt.Scale(domain=[float(np.floor(allnl.prod_gbp.min() - 1)), float(np.ceil(allnl.prod_gbp.max() + 1))], zero=False, nice=False)
YSN = alt.Scale(domain=[float(np.floor(allnl.realloc.min() - 1)), float(np.ceil(allnl.realloc.max() + 1))], zero=False, nice=False)
SIZEN = alt.Scale(domain=[0, float(allnl.emp.max())], range=[8, 200])
WN, HN = W4, H4

def nl_rates(yrs, ni_from):
    j = ni_trim(jf[jf.year.between(*yrs)], ni_from); j = j[~LONDON(j)]
    pj = ni_trim(prod[prod.year.between(*yrs)], ni_from); pj = pj[~LONDON(pj)]
    return 100 * j[["total_jc", "total_jd"]].sum().sum() / j.emp_avg.sum(), pj.gva_real.sum() / pj.hours_worked_weekly.sum()

ABBR = {"Berkshire, Buckinghamshire and Oxfordshire": "Berks, Bucks & Oxon", "Bedfordshire and Hertfordshire": "Beds & Herts",
        "Gloucestershire, Wiltshire and Bath/Bristol area": "Glos, Wilts & Bristol", "Leicestershire, Rutland and Northamptonshire": "Leics & Northants",
        "Herefordshire, Worcestershire and Warwickshire": "Herefs, Worcs & Warks", "Surrey, East and West Sussex": "Surrey & Sussex",
        "East Yorkshire and Northern Lincolnshire": "E Yorks & N Lincs", "Northumberland, and Tyne and Wear": "Tyneside & Northumb.",
        "Derbyshire and Nottinghamshire": "Derbys & Notts", "Shropshire and Staffordshire": "Shrops & Staffs",
        "West Wales and The Valleys": "West Wales & Valleys", "Cornwall and Isles of Scilly": "Cornwall", "Tees Valley and Durham": "Tees Valley & Durham",
        "Hampshire and Isle of Wight": "Hampshire & IoW", "Dorset and Somerset": "Dorset & Somerset", "Highlands and Islands": "Highlands & Islands"}

NATIONS = ["England", "Wales", "Scotland", "Northern Ireland"]
NCOLS = [ECO["blue-light"], ECO["red"], ECO["blue-dark"], ECO["yellow"]]

def panel_nl(lab, first):
    d = nl_panels[lab].assign(full_name=lambda x: x.itl2_name, itl2_name=lambda x: x.itl2_name.replace(ABBR),
                                nation=lambda x: x.itl2_code.str[:3].map({"TLL": "Wales", "TLM": "Scotland", "TLN": "Northern Ireland"}).fillna("England")); ukr, ukp = nl_rates(wins_c[lab], ni_c[lab])
    r = np.corrcoef(d.realloc, d.prod_gbp)[0, 1]
    labs = place_labels(d, XSN, YSN, WN, H4, size=SIZEN, xcol="prod_gbp", max_labels=12, max_len=22, force=False, pad=2)
    ax_y = alt.Axis(title="Reallocation rate (% of employment)" if first else None, labels=first, ticks=first, labelExpr="datum.label + '%'")
    return alt.layer(
        alt.Chart(pd.DataFrame({"v": [ukp]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(x=alt.X("v:Q", scale=XSN)),
        alt.Chart(pd.DataFrame({"v": [ukr]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YSN, axis=ax_y)),
        fit_layer(d, "prod_gbp", XSN, GREY).encode(y=alt.Y("y:Q", scale=YSN)),
        alt.Chart(d).mark_circle(opacity=0.9, stroke="white", strokeWidth=0.6).encode(
            color=alt.Color("nation:N", scale=alt.Scale(domain=NATIONS, range=NCOLS), legend=alt.Legend(title=None, orient="bottom") if first else None),
            x=alt.X("prod_gbp:Q", scale=XSN, title="GVA per hour worked (£, 2022 prices)",
                    axis=alt.Axis(values=list(range(int(XSN.domain[0]) // 5 * 5 + 5, int(XSN.domain[1]) + 1, 5)), labelExpr="'£' + datum.label")),
            y=alt.Y("realloc:Q", scale=YSN, axis=ax_y), size=alt.Size("emp:Q", scale=SIZEN, legend=None),
            order=alt.Order("emp:Q", sort="descending"),
            tooltip=[alt.Tooltip("full_name:N", title="Region"), alt.Tooltip("prod_gbp:Q", format=".1f", title="GVA per hour (£)"),
                     alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)"), alt.Tooltip("emp:Q", format=",.0f", title="Employment")]),
        *[alt.Chart(labs.iloc[[i]]).mark_text(align=r_["align"], baseline="middle", dx=r_["dx"], dy=r_["dy"], fontSize=9, color=TXT)
            .encode(x=alt.X("prod_gbp:Q", scale=XSN), y=alt.Y("realloc:Q", scale=YSN), text="short:N") for i, r_ in labs.iterrows()],
    ).properties(width=WN, height=HN, title=alt.Title(lab, subtitle=f"r = {r:.2f} · average {ukr:.1f}%, £{ukp:.0f} an hour", anchor="start", fontSize=13, subtitleFontSize=11))

pcn = alt.hconcat(*[panel_nl(lab, i == 0) for i, lab in enumerate(wins_c)], spacing=12).resolve_scale(color="shared").properties(title=alt.Title(
    "Outside London, more productive regions churn only slightly more",
    subtitle=["Reallocation rate against real GVA per hour, ITL2 regions outside London, pooled over each period. Dot size = employment; line = linear fit.",
              "Dotted lines = average of all regions outside London (values under each period). Real GVA per hour in 2022 prices (ONS chained-volume). Northern Ireland excluded in 2004–10.", SRC2]))
styles.save(pcn, str(FIG), "reallocation_productivity_windows_C_excl_london", width=WN, height=HN)
print({lab: round(np.corrcoef(d.realloc, d.prod_gbp)[0, 1], 2) for lab, d in nl_panels.items()})
pcn

{'2004–10': np.float64(0.32), '2011–17': np.float64(0.35), '2018–23': np.float64(0.24)}


alt.HConcatChart(...)

### Change in productivity vs change in reallocation

Like the ageing chart: did regions where productivity grew more see churn fall less (or more)? For each region, the change between two pooled windows in (i) **real GVA per hour** (% change, chained volume, 2022 £) and (ii) the **reallocation rate** (percentage points).
- **Whole period:** 2004–08 → 2019–23 (five-year windows at each end; productivity data end in 2023).
- **Three timeframes:** the step from 2004–10 to 2011–17, and from 2011–17 to 2018–23 (window set C).

Northern Ireland is excluded wherever the earlier window is before 2011.

In [8]:
from chart_setup import place_labels as place_labels_g
def change_frame(early, late, ni_e, ni_l):
    e = pooled_realloc(early, ni_from=ni_e).join(prod_gbp(early, ni_e), on="itl2_code")
    l = pooled_realloc(late, ni_from=ni_l).join(prod_gbp(late, ni_l), on="itl2_code")
    d = pd.DataFrame({"d_realloc": l.realloc - e.realloc, "d_prod": 100 * (l.prod_gbp / e.prod_gbp - 1)}).dropna().reset_index()
    d = d[d.itl2_code != "TLN0"] if ni_e == 9999 else d
    d["itl2_name"] = d.itl2_name.replace(ABBR)
    d["group"] = np.where(LONDON(d), "London", "Rest of UK")
    return d

STEPS = {"whole": [("2004–08 → 2019–23", (2004, 2008), (2019, 2023), 9999, None)],
         "steps": [("2004–10 → 2011–17", (2004, 2010), (2011, 2017), 9999, None), ("2011–17 → 2018–23", (2011, 2017), (2018, 2023), None, None)]}
chg = {lab: change_frame(e, l, ne, nl_) for v in STEPS.values() for lab, e, l, ne, nl_ in v}
for lab, d in chg.items():
    o = d[~LONDON(d)]
    print(f"{lab}: n={len(d)}  r={np.corrcoef(d.d_prod, d.d_realloc)[0, 1]:+.2f}  excl. London r={np.corrcoef(o.d_prod, o.d_realloc)[0, 1]:+.2f}  "
          f"rho={stats.spearmanr(d.d_prod, d.d_realloc)[0]:+.2f} | mean Δrealloc {d.d_realloc.mean():+.1f} pp, mean Δprod {d.d_prod.mean():+.1f}%")

def scales(labs_):
    a = pd.concat([chg[l] for l in labs_])
    return (alt.Scale(domain=[float(np.floor(a.d_prod.min() / 5) * 5 - 2), float(np.ceil(a.d_prod.max() / 5) * 5 + 2)], nice=False),
            alt.Scale(domain=[float(np.floor(min(a.d_realloc.min(), 0) - 1)), float(np.ceil(max(a.d_realloc.max(), 0) + 1))], nice=False))

def change_panel(lab, first, W, H, XSC, YSC):
    d = chg[lab]; o = d[~LONDON(d)]
    r_all, r_ex = np.corrcoef(d.d_prod, d.d_realloc)[0, 1], np.corrcoef(o.d_prod, o.d_realloc)[0, 1]
    labs = place_labels_g(d, "d_prod", "d_realloc", XSC, YSC, W, H, max_labels=8, max_len=22, pad=2)
    ax_y = alt.Axis(title="Change in reallocation rate (percentage points)" if first else None, labels=first, ticks=first, labelFlush=True,
                    labelExpr="datum.value > 0 ? '+' + datum.label : datum.label")
    return alt.layer(
        alt.Chart(pd.DataFrame({"v": [0.0]})).mark_rule(color=TXT, opacity=0.5).encode(x=alt.X("v:Q", scale=XSC)),
        alt.Chart(pd.DataFrame({"v": [0.0]})).mark_rule(color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YSC, axis=ax_y)),
        fit_layer(d.rename(columns={"d_realloc": "realloc"}), "d_prod", XSC, ECO["red"]).encode(y=alt.Y("y:Q", scale=YSC)),
        fit_layer(o.rename(columns={"d_realloc": "realloc"}), "d_prod", XSC, ECO["blue-dark"]).encode(y=alt.Y("y:Q", scale=YSC)),
        alt.Chart(d).mark_circle(size=70, opacity=0.9, stroke="white", strokeWidth=0.6).encode(
            x=alt.X("d_prod:Q", scale=XSC, title="Change in real GVA per hour (%)", axis=alt.Axis(labelExpr="datum.value > 0 ? '+' + datum.label + '%' : datum.label + '%'")),
            y=alt.Y("d_realloc:Q", scale=YSC, axis=ax_y),
            color=alt.Color("group:N", scale=alt.Scale(domain=["London", "Rest of UK"], range=[ECO["red"], ECO["blue-light"]]),
                            legend=alt.Legend(title=None, orient="bottom") if first else None),
            tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("d_prod:Q", format="+.1f", title="Change in GVA/hour (%)"),
                     alt.Tooltip("d_realloc:Q", format="+.1f", title="Change in reallocation (pp)")]),
        *[alt.Chart(labs.iloc[[i]]).mark_text(align=r_["align"], dx=r_["dx"], fontSize=9, color=TXT)
            .encode(x=alt.X("d_prod:Q", scale=XSC), y=alt.Y("d_realloc:Q", scale=YSC), text="short:N") for i, r_ in labs.iterrows()],
    ).properties(width=W, height=H, title=alt.Title(lab, subtitle=f"r = {r_all:.2f} (excl. London {r_ex:.2f})", anchor="start", fontSize=13, subtitleFontSize=11))

SUB2 = "Each dot is an ITL2 region; changes between pooled windows. Red line = fit across all regions, navy = excluding London. Northern Ireland excluded where the earlier window starts before 2011."
whole = change_panel("2004–08 → 2019–23", True, 520, 380, *scales(["2004–08 → 2019–23"])).properties(title=alt.Title(
    "Churn fell almost everywhere, whatever the pace of productivity growth",
    subtitle=["Change in real GVA per hour (chained volume, 2022 prices) against change in the job reallocation rate, 2004–08 to 2019–23.", SUB2, SRC2]))
styles.save(whole, str(FIG), "reallocation_productivity_change_whole", width=520, height=380)
SC_STEPS = scales([lab for lab, *_ in STEPS["steps"]])
steps = alt.hconcat(*[change_panel(lab, i == 0, 330, 340, *SC_STEPS) for i, (lab, *_) in enumerate(STEPS["steps"])], spacing=14).resolve_scale(color="shared").properties(
    title=alt.Title("Productivity growth and the change in churn are barely related in either period",
    subtitle=["Change in real GVA per hour against change in the job reallocation rate between consecutive periods (2004–10, 2011–17, 2018–23).", SUB2, SRC2]))
styles.save(steps, str(FIG), "reallocation_productivity_change_steps", width=330, height=340)
whole.properties(width=520, height=380)

2004–08 → 2019–23: n=40  r=-0.34  excl. London r=-0.19  rho=-0.28 | mean Δrealloc -5.2 pp, mean Δprod +8.4%
2004–10 → 2011–17: n=40  r=-0.19  excl. London r=-0.11  rho=-0.23 | mean Δrealloc -3.7 pp, mean Δprod +3.2%
2011–17 → 2018–23: n=41  r=+0.14  excl. London r=+0.17  rho=+0.16 | mean Δrealloc -0.9 pp, mean Δprod +4.5%


alt.LayerChart(...)

In [9]:
# Robustness: change charts with GDP-deflated GVA per hour (one UK-wide deflator) instead of ONS chained volume
def change_frame_defl(early, late, ni_e, ni_l):
    d = change_frame(early, late, ni_e, ni_l).set_index("itl2_code")
    e, l = prod_defl(early, ni_e), prod_defl(late, ni_l)
    d["d_prod_cv"] = d.d_prod
    d["d_prod"] = 100 * (l / e - 1).reindex(d.index)
    return d.reset_index()

chg_cv = dict(chg)
chg = {lab: change_frame_defl(e, l, ne, nl_) for v in STEPS.values() for lab, e, l, ne, nl_ in v}
cmp = []
for lab, d in chg.items():
    o = d[~LONDON(d)]; lon = d[LONDON(d)]
    cmp.append({"windows": lab, "r (GDP deflator)": np.corrcoef(d.d_prod, d.d_realloc)[0, 1],
                "r (chained volume)": np.corrcoef(d.d_prod_cv, d.d_realloc)[0, 1],
                "excl. London r (GDP defl.)": np.corrcoef(o.d_prod, o.d_realloc)[0, 1],
                "London avg growth % (GDP defl.)": lon.d_prod.mean(), "London avg growth % (chained vol.)": lon.d_prod_cv.mean(),
                "Rest avg growth % (GDP defl.)": o.d_prod.mean(), "Rest avg growth % (chained vol.)": o.d_prod_cv.mean()})
print(pd.DataFrame(cmp).round(2).to_string(index=False))
print(chg["2004–08 → 2019–23"][LONDON(chg["2004–08 → 2019–23"])][["itl2_name", "d_prod_cv", "d_prod", "d_realloc"]].round(1).to_string(index=False))

SUB3 = "GVA per hour in current prices deflated by the UK GDP deflator (2022 prices). Red line = fit across all regions, navy = excluding London. NI excluded where the earlier window starts before 2011."
whole_d = change_panel("2004–08 → 2019–23", True, 520, 380, *scales(["2004–08 → 2019–23"])).properties(title=alt.Title(
    "With a single UK price index the picture is the same: London's productivity lagged and churn fell everywhere",
    subtitle=["Change in GVA per hour (GDP-deflated) against change in the job reallocation rate, 2004–08 to 2019–23. Robustness check on the chained-volume version.", SUB3, SRC2]))
styles.save(whole_d, str(FIG), "reallocation_productivity_change_whole_gdpdefl", width=520, height=380)
SC_D = scales([lab for lab, *_ in STEPS["steps"]])
steps_d = alt.hconcat(*[change_panel(lab, i == 0, 330, 340, *SC_D) for i, (lab, *_) in enumerate(STEPS["steps"])], spacing=14).resolve_scale(color="shared").properties(
    title=alt.Title("Productivity growth and the change in churn, GDP-deflated",
    subtitle=["Change in GVA per hour (GDP-deflated) against change in the job reallocation rate between consecutive periods.", SUB3, SRC2]))
styles.save(steps_d, str(FIG), "reallocation_productivity_change_steps_gdpdefl", width=330, height=340)
chg = chg_cv                                                       # restore the main (chained-volume) version
whole_d.properties(width=520, height=380)

          windows  r (GDP deflator)  r (chained volume)  excl. London r (GDP defl.)  London avg growth % (GDP defl.)  London avg growth % (chained vol.)  Rest avg growth % (GDP defl.)  Rest avg growth % (chained vol.)
2004–08 → 2019–23             -0.37               -0.34                       -0.28                            -2.17                               -1.12                           7.12                              9.76
2004–10 → 2011–17             -0.21               -0.19                       -0.22                            -0.28                                1.36                           1.02                              3.43
2011–17 → 2018–23              0.11                0.14                        0.12                            -1.54                               -2.26                           5.45                              5.43
                         itl2_name  d_prod_cv  d_prod  d_realloc
               Inner London - West        6.9    10.3       -3.

alt.LayerChart(...)

In [10]:
# Index line chart: UK churn and real productivity over time, 2004-08 = 100 (productivity_reallocation_index)
yrs = range(2004, 2024)
uj = jf[jf.year.isin(yrs) & (jf.itl2_code != "TLN0")].groupby("year")[["total_jc", "total_jd", "emp_avg"]].sum()   # NI excluded (no data before 2011) for a consistent series
up = prod[prod.year.isin(yrs)].groupby("year")[["gva_real", "hours_worked_weekly"]].sum()
ts = pd.DataFrame({"Job reallocation rate": 100 * (uj.total_jc + uj.total_jd) / uj.emp_avg, "Real GVA per hour": up.gva_real / up.hours_worked_weekly})
idx = 100 * ts / ts.loc[2004]
print(pd.concat([ts.round(1), idx.round(0).add_suffix(" (index)")], axis=1).loc[[2004, 2007, 2008, 2009, 2011, 2015, 2019, 2023]].to_string())
long = idx.reset_index(names="year").melt(id_vars="year", var_name="series", value_name="index").merge(
    ts.reset_index(names="year").melt(id_vars="year", var_name="series", value_name="level"), on=["year", "series"])
SER = ["Real GVA per hour", "Job reallocation rate"]; SCOL = [ECO["blue-light"], ECO["blue-dark"]]
XI, YI = alt.Scale(domain=[2004, 2023], nice=False), alt.Scale(domain=[60, 125], nice=False)
shade = pd.DataFrame({"x0": [2008, 2020], "x1": [2009.99, 2021.99], "t": ["Financial crisis", "COVID"]})
end = long[long.year == 2023]
idx_ch = alt.layer(
    alt.Chart(shade).mark_rect(color=GREY, opacity=0.15).encode(x=alt.X("x0:Q", scale=XI), x2="x1:Q"),
    alt.Chart(shade).mark_text(align="left", baseline="top", dx=3, dy=4, fontSize=10, color=TXT).encode(x=alt.X("x0:Q", scale=XI), y=alt.value(0), text="t:N"),
    alt.Chart(pd.DataFrame({"v": [100.0]})).mark_rule(color=TXT, opacity=0.4).encode(y=alt.Y("v:Q", scale=YI)),
    alt.Chart(long).mark_line(strokeWidth=3).encode(
        x=alt.X("year:Q", scale=XI, title=None, axis=alt.Axis(format="d", values=list(range(2004, 2024, 2)))),
        y=alt.Y("index:Q", scale=YI, title="Index (2004 = 100)"),
        color=alt.Color("series:N", scale=alt.Scale(domain=SER, range=SCOL), legend=None),
        tooltip=["series:N", "year:Q", alt.Tooltip("level:Q", format=".1f", title="Level (% or £)"), alt.Tooltip("index:Q", format=".0f")]),
    alt.Chart(end).mark_text(align="left", dx=8, fontSize=12, fontWeight="bold").encode(
        x=alt.X("year:Q", scale=XI), y=alt.Y("index:Q", scale=YI), text="series:N",
        color=alt.Color("series:N", scale=alt.Scale(domain=SER, range=SCOL), legend=None)),
).properties(title=alt.Title("Since the financial crisis, job churn has run about a fifth below its 2004 level, while productivity growth slowed",
    subtitle=[f"UK, index (2004 = 100). In 2004 the reallocation rate was {ts.loc[2004].iloc[0]:.1f}% and real GVA per hour £{ts.loc[2004].iloc[1]:.0f} (2022 prices).",
              "Northern Ireland excluded (job flows from 2011). 2010 and 2023 are known spikes in the business register data.", SRC2]))
styles.save(idx_ch, str(FIG), "productivity_reallocation_index", width=560, height=340)
idx_ch.properties(width=560, height=340)

      Job reallocation rate  Real GVA per hour  Job reallocation rate (index)  Real GVA per hour (index)
year                                                                                                    
2004                   35.9               35.7                          100.0                      100.0
2007                   30.9               37.6                           86.0                      105.0
2008                   30.4               37.8                           85.0                      106.0
2009                   29.0               36.8                           81.0                      103.0
2011                   26.8               38.0                           75.0                      106.0
2015                   28.6               38.8                           80.0                      109.0
2019                   27.9               39.6                           78.0                      111.0
2023                   30.9               40.8         

alt.LayerChart(...)

### Change since before the financial crisis

Both panels share one **pre-crisis baseline, 2004–08** (five years, matching the comparison windows; 2004 is the first year with hours data and the recession only began in mid-2008). Compared with:
- **2011–15:** the years soon after the crisis (starting in 2011 avoids the 2010 LBD spike and the recession itself);
- **2019–23:** the latest years with productivity data.

Productivity: % change in real GVA per hour (chained volume). Churn: change in the reallocation rate (pp). Northern Ireland is excluded (no reliable job flows before 2011). No fit lines: the correlations are weak, so the subtitle reports them instead. (change_precrisis)

In [11]:
PRE = (2004, 2008)
POST = {"2004–08 → 2011–15 (soon after the crisis)": (2011, 2015), "2004–08 → 2019–23 (latest)": (2019, 2023)}
chg_cv2 = dict(chg)
chg = {lab: change_frame(PRE, yrs, 9999, None) for lab, yrs in POST.items()}
rows = []
for lab, d in chg.items():
    o = d[~LONDON(d)]
    rows.append({"comparison": lab, "n": len(d), "avg Δ churn (pp)": d.d_realloc.mean(), "avg Δ GVA/hour (%)": d.d_prod.mean(),
                 "regions with lower churn": int((d.d_realloc < 0).sum()), "r": np.corrcoef(d.d_prod, d.d_realloc)[0, 1],
                 "r excl. London": np.corrcoef(o.d_prod, o.d_realloc)[0, 1], "rho": stats.spearmanr(d.d_prod, d.d_realloc)[0]})
print(pd.DataFrame(rows).round(2).to_string(index=False))

AREAS = ["London", "Rest of England", "Wales", "Scotland", "Northern Ireland"]
ACOLS = [ECO["red"], ECO["blue-light"], ECO["turquoise"], ECO["blue-dark"], ECO["yellow"]]
def area(d):
    return np.where(LONDON(d), "London", d.itl2_code.str[:3].map({"TLL": "Wales", "TLM": "Scotland", "TLN": "Northern Ireland"}).fillna("Rest of England"))

def change_panel_plain(lab, first, W, H, XSC, YSC):
    d = chg[lab].assign(area=lambda x: area(x)); o = d[~LONDON(d)]
    present = [a for a in AREAS if a in set(pd.concat(chg.values()).pipe(area))]
    r_all, r_ex = np.corrcoef(d.d_prod, d.d_realloc)[0, 1], np.corrcoef(o.d_prod, o.d_realloc)[0, 1]
    labs = place_labels_g(d, "d_prod", "d_realloc", XSC, YSC, W, H, max_labels=8, max_len=22, pad=2)
    ax_y = alt.Axis(title="Change in reallocation rate" if first else None, labels=first, ticks=first, labelFlush=True,
                    labelExpr="datum.value > 0 ? '+' + datum.label + 'pp' : datum.value < 0 ? datum.label + 'pp' : datum.label")
    title = lab.split(" (")[0]
    bb = np.polyfit(d.d_prod, d.d_realloc, 1); fx = np.array([d.d_prod.min(), d.d_prod.max()])
    top_lon = d[LONDON(d)].nlargest(1, "d_realloc") if first else d.iloc[0:0]
    labs = labs[~labs.itl2_code.isin(top_lon.itl2_code)] if len(labs) else labs
    return alt.layer(
        alt.Chart(pd.DataFrame({"v": [0.0]})).mark_rule(color=TXT, opacity=0.5).encode(x=alt.X("v:Q", scale=XSC)),
        alt.Chart(pd.DataFrame({"v": [0.0]})).mark_rule(color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YSC, axis=ax_y)),
        alt.Chart(pd.DataFrame({"x": fx, "y": np.polyval(bb, fx)})).mark_line(color=GREY, strokeWidth=1.6, opacity=0.6)
           .encode(x=alt.X("x:Q", scale=XSC), y=alt.Y("y:Q", scale=YSC)),
        alt.Chart(d).mark_circle(size=70, opacity=0.9, stroke="white", strokeWidth=0.6).encode(
            x=alt.X("d_prod:Q", scale=XSC, title="Change in real GVA per hour", axis=alt.Axis(labelExpr="datum.value > 0 ? '+' + datum.label + '%' : datum.label + '%'")),
            y=alt.Y("d_realloc:Q", scale=YSC, axis=ax_y),
            color=alt.Color("area:N", scale=alt.Scale(domain=present, range=[ACOLS[AREAS.index(a)] for a in present]),
                            legend=alt.Legend(title=None, orient="bottom") if first else None),
            tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("area:N", title="Area"), alt.Tooltip("d_prod:Q", format="+.1f", title="Change in GVA/hour (%)"),
                     alt.Tooltip("d_realloc:Q", format="+.1f", title="Change in reallocation (pp)")]),
        *[alt.Chart(labs.iloc[[i]]).mark_text(align=r_["align"], dx=r_["dx"], fontSize=9, color=TXT)
            .encode(x=alt.X("d_prod:Q", scale=XSC), y=alt.Y("d_realloc:Q", scale=YSC), text="short:N") for i, r_ in labs.iterrows()],
        *[alt.Chart(top_lon.assign(short=top_lon.itl2_name.str.replace(" - ", " ").str.replace("East and North East", "East & North East")
                                                   .str.replace("West and North West", "West & North West")))
             .mark_text(align="left", dx=8, fontSize=9, color=TXT)
             .encode(x=alt.X("d_prod:Q", scale=XSC), y=alt.Y("d_realloc:Q", scale=YSC), text="short:N")] if len(top_lon) else [],
    ).properties(width=W, height=H, title=alt.Title(title, anchor="start", fontSize=13))

SC_P = scales(list(POST))
pre_ch = alt.hconcat(*[change_panel_plain(lab, i == 0, 350, 350, *SC_P) for i, lab in enumerate(POST)], spacing=16).resolve_scale(color="shared").properties(
    title=alt.Title("Since before the financial crisis, churn has fallen in almost every region, whatever its productivity growth",
    subtitle=["Change in real GVA per hour (chained volume, 2022 prices) and in the job reallocation rate, each against the pre-crisis years 2004–08.",
              "Each dot is an ITL2 region; grey line = fit across all regions. Results similar excluding 2020–21 (COVID). Northern Ireland excluded (no data before 2011).", SRC2]))
styles.save(pre_ch, str(FIG), "reallocation_productivity_change_precrisis", width=350, height=350)
chg = chg_cv2
pre_ch

                               comparison  n  avg Δ churn (pp)  avg Δ GVA/hour (%)  regions with lower churn     r  r excl. London   rho
2004–08 → 2011–15 (soon after the crisis) 40             -3.90                2.89                        39 -0.17           -0.11 -0.07
               2004–08 → 2019–23 (latest) 40             -5.21                8.40                        40 -0.34           -0.19 -0.28


alt.HConcatChart(...)

### 1e. What this shows (and doesn't)

- **Levels:** regions with higher GVA per hour have higher reallocation. Across all 41 regions, 10% higher productivity goes with about 1.4 pp more reallocation (r ≈ 0.66). The link survives dropping COVID years, using pre-COVID or 2004–08 windows, and employment weighting.
- **But the link is mostly London.** Excluding London's five regions cuts the slope by roughly two-thirds (r ≈ 0.25, only marginally significant). Outside London, productivity and churn are only weakly related.
- **Growth:** reallocation does **not** predict later productivity growth, whether across regions (2004–07 churn → 2007–19 growth, r ≈ −0.2, not significant, and close to zero once initial productivity or London is controlled for) or within regions over time (fixed-effects panel; 1- and 3-year horizons are small and insignificant). If anything, the later window (2011–15 churn → 2015–23 growth) is mildly negative (r ≈ −0.3). This is consistent with the literature: what matters is *where* jobs move (towards more productive firms), not how much churn there is.
- **Fit shape:** the charts use straight-line fits and correlations on GVA per hour itself. The tables in (a) use log GVA per hour, so slopes read as "per 10% higher productivity". A straight line fits as well as the log (R² slightly higher with London, identical without), so the conclusions don't depend on the choice.
- **Caveats:**
  - **Sector mix** may drive both measures, e.g. London's finance and business services. A within-industry version needs the LBD by sector, run in the secure environment.
  - **Price levels:** nominal GVA per hour includes regional price levels, so London's lead is partly prices.
  - **Workplace vs residence:** GVA and hours are workplace-based while LBD employment is site-based. Both measures are by place of work, so they are consistent, but commuting into Inner London inflates both.
  - **Small samples:** about 40 regions, so the confidence intervals are wide.
  - **Next step:** *productivity-enhancing reallocation* needs firm-level productivity, i.e. a Foster–Haltiwanger–Krizan or Olley–Pakes decomposition. That also has to be done in the secure environment.